In [1]:
import cv2
import json
import numpy as np
import os
import pathlib
from glob import glob

In [2]:
def load_calibration(calibration_file, key):
    with open(calibration_file, 'r') as f:
        data = json.load(f)

    return np.array(data['cams'][key]['calibration']['cameraMatrix']), np.array(data['cams'][key]['calibration']['distCoeffs'])

def load_stereo_calibration(calibration_file):
    with open(calibration_file, 'r') as f:
        data = json.load(f)

    return (np.array(data['stereoCalib']['0_1']['rotationMatrix']),
            np.array(data['stereoCalib']['0_1']['translationMatrix']),
            np.array(data['stereoCalib']['0_1']['essentialMatrix']),
            np.array(data['stereoCalib']['0_1']['fundamentalMatrix']))

def on_mouse_left(event, x, y, flags, userdata):
    if event == cv2.EVENT_MOUSEMOVE:
        pt, _ = transfer_bbox_right_left_dist(camera_matrix_l, dist_coefs_l, camera_matrix_r, dist_coefs_r, R, T, Z, (x, y))
        userdata['mouse_pos_l'] = tuple(pt.reshape(2).astype(np.int64))
        userdata['mouse_pos_r'] = (x, y)

In [3]:
def transfer_bbox_right_left_dist(cam_matrix_l, dist_coefs_l, cam_matrix_r, dist_coefs_r, rotation, translation, Z, point):
    fx = cam_matrix_r[0, 0]
    fy = cam_matrix_r[1, 1]
    cx = cam_matrix_r[0, 2]
    cy = cam_matrix_r[1, 2]

    pt = np.array([[[point[0], point[1]]]], dtype=np.float32)
    pt_ud = cv2.undistortPoints(pt, camera_matrix_r, dist_coefs_r, P=camera_matrix_r)
    u_ud, v_ud = pt_ud.reshape(2)

    X = (u_ud - cx) * Z / fx
    Y = (v_ud - cy) * Z / fy

    X_R = np.array([[X], [Y], [Z]], dtype=np.float64)  # (3,1)
    translation = translation.reshape(3, 1).astype(np.float64)
    rotation = rotation.astype(np.float64)

    X_L = rotation.T @ (X_R - translation)

    return cv2.projectPoints(
        X_L.T,
        np.zeros((3, 1), dtype=np.float64),
        np.zeros((3, 1), dtype=np.float64),
        cam_matrix_l,
        dist_coefs_l
    )

In [4]:
def load_bbox_coords(file):
    bbox_coords = []

    with open(file) as f:
        while line := f.readline():
            bbox = line.rstrip().split(" ")
            bbox_coords.append(bbox)

    return np.array(bbox_coords).astype(float)

In [5]:
def convert_xywhn_xyxy(size, box):
    x1 = (box[0] - box[2] / 2) * size[0]
    y1 = (box[1] - box[3] / 2) * size[1]
    x2 = (box[0] + box[2] / 2) * size[0]
    y2 = (box[1] + box[3] / 2) * size[1]

    return [x1, y1, x2, y2]

In [20]:
data_dir = pathlib.Path(R"D:\datasets\own_data\mason_bees_6")
img_output = pathlib.Path(R"./datasets/transfers")
job_json = r"D:\datasets\own_data\calibration_mason_bee_5_6\job_data.json"

image = cv2.imread(glob(str(data_dir / "processed" / "cam_0" / "frame_*"))[0])
shape_0 = image.shape[0:2][::-1]
image = cv2.imread(glob(str(data_dir / "processed" / "cam_1" / "frame_*"))[0])
shape_1 = image.shape[0:2][::-1]

Z = 263/1000  # in meters

camera_matrix_l, dist_coefs_l = load_calibration(job_json, "cam_0")
camera_matrix_r, dist_coefs_r = load_calibration(job_json, "cam_1")
R, T, E, F = load_stereo_calibration(job_json)

frame_num = "frame_1.png"

cam_0 = cv2.imread(str(data_dir / "processed" / "cam_0" / frame_num))
cam_1 = cv2.imread(str(data_dir / "processed" / "cam_1" / frame_num))

callback_state = {"mouse_pos_l": (-1, -1), "mouse_pos_r": (-1, -1)}
win_name_0 = "left"
cv2.namedWindow(win_name_0)

win_name_1 = "right"
cv2.namedWindow(win_name_1)
cv2.setMouseCallback(win_name_1, on_mouse_left, callback_state)

while True:
    display_0 = cam_0.copy()
    display_1 = cam_1.copy()

    if 0 <= callback_state['mouse_pos_l'][0] <= shape_0[0] and 0 <= callback_state['mouse_pos_l'][1] <= shape_0[1]:
        text_0 = f"x={callback_state['mouse_pos_l'][0]}, y={callback_state['mouse_pos_l'][1]}"
        cv2.putText(
            display_0,
            text_0,
            (10, 25),  # top-left corner
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
        cv2.circle(display_0, callback_state['mouse_pos_l'], 4, (255,0,255), -1)

    if callback_state['mouse_pos_r'][0] >= 0 and callback_state['mouse_pos_r'][1] >= 0:
        text_1 = f"x={callback_state['mouse_pos_r'][0]}, y={callback_state['mouse_pos_r'][1]}"
        cv2.putText(
            display_1,
            text_1,
            (10, 25),  # top-left corner
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

    cv2.imshow(win_name_0, display_0)
    cv2.imshow(win_name_1, display_1)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cv2.destroyAllWindows()


In [6]:
data_dir = pathlib.Path(R"C:\Users\Bas_K\OneDrive\Documenten\School\Maastricht university\wafer_research\datasets\silicon_test_3")
# img_output = pathlib.Path(R"./datasets/transfers")
job_json = r"C:\Users\Bas_K\OneDrive\Documenten\School\Maastricht university\wafer_research\datasets\calibration_2_3\job_data.json"

image = cv2.imread(glob(str(data_dir / "processed" / "cam_1" / "frame_*"))[0])
shape = image.shape[0:2][::-1]

bbox_files = glob(str(data_dir / "prediction" / "frame_*.txt"))

Z = 408/1000  # in meters

for bbox_file in bbox_files:
    bbox_coords = load_bbox_coords(bbox_file)
    bbox_denormalised = []
    for bbox_coord in bbox_coords:
        bbox = convert_xywhn_xyxy(shape, bbox_coord[1:])
        bbox.insert(0, bbox_coord[0])
        bbox_denormalised.append(bbox)

    bbox_denormalised = np.array(bbox_denormalised, dtype=np.int64)

    camera_matrix_l, dist_coefs_l = load_calibration(job_json, "cam_0")
    camera_matrix_r, dist_coefs_r = load_calibration(job_json, "cam_1")
    R, T, E, F = load_stereo_calibration(job_json)

    cam_0 = cv2.imread(str(data_dir / "processed" / "cam_0" / pathlib.Path(bbox_file).with_suffix(".png").name))
    cam_1 = cv2.imread(str(data_dir / "processed" / "cam_1" / pathlib.Path(bbox_file).with_suffix(".png").name))

    callback_state = {"mouse_pos_l": (-1, -1), "mouse_pos_r": (-1, -1)}
    win_name_0 = "left"
    cv2.namedWindow(win_name_0)

    win_name_1 = "right"
    cv2.namedWindow(win_name_1)
    cv2.setMouseCallback(win_name_1, on_mouse_left, callback_state)

    for row in bbox_denormalised:
        point_r_1 = (int(row[1]), int(row[2]))
        point_r_2 = (int(row[3]), int(row[4]))
        cv2.rectangle(cam_1, point_r_1, point_r_2, (0, 0, 255), 2)

        img_pt, _ = transfer_bbox_right_left_dist(camera_matrix_l, dist_coefs_l, camera_matrix_r, dist_coefs_r, R, T, Z, point_r_1)
        point_l_1 = tuple(img_pt.reshape(2).astype(np.int64))


        img_pt, _ = transfer_bbox_right_left_dist(camera_matrix_l, dist_coefs_l, camera_matrix_r, dist_coefs_r, R, T, Z, point_r_2)
        point_l_2 = tuple(img_pt.reshape(2).astype(np.int64))


        cv2.rectangle(cam_0, point_l_1, point_l_2, (0, 0, 255), 2)


    while True:
        display_0 = cam_0.copy()
        display_1 = cam_1.copy()

        if callback_state['mouse_pos_l'][0] >= 0 and callback_state['mouse_pos_l'][1] >= 0:
            text_0 = f"x={callback_state['mouse_pos_l'][0]}, y={callback_state['mouse_pos_l'][1]}"
            cv2.putText(
                display_0,
                text_0,
                (10, 25),  # top-left corner
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )
            cv2.circle(display_0, callback_state['mouse_pos_l'], 4, (255,0,255), -1)

        if callback_state['mouse_pos_r'][0] >= 0 and callback_state['mouse_pos_r'][1] >= 0:
            text_1 = f"x={callback_state['mouse_pos_r'][0]}, y={callback_state['mouse_pos_r'][1]}"
            cv2.putText(
                display_1,
                text_1,
                (10, 25),  # top-left corner
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

        cv2.imshow(win_name_0, display_0)
        cv2.imshow(win_name_1, display_1)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cv2.destroyAllWindows()


In [ ]:
# data_dir = pathlib.Path(R"D:\datasets\own_data\mason_bees_6")
# img_output = pathlib.Path(R"./datasets/transfers")
# job_json = r"D:\datasets\own_data\calibration_mason_bee_5_6\job_data.json"

data_dir = pathlib.Path(R"C:\Users\Bas_K\OneDrive\Documenten\School\Maastricht university\wafer_research\datasets\silicon_test_3")
# img_output = pathlib.Path(R"./datasets/transfers")
job_json = r"C:\Users\Bas_K\OneDrive\Documenten\School\Maastricht university\wafer_research\datasets\calibration_2_3\job_data.json"

image = cv2.imread(glob(str(data_dir / "processed" / "cam_0" / "frame_*"))[0])
shape_0 = image.shape[0:2][::-1]
image = cv2.imread(glob(str(data_dir / "processed" / "cam_1" / "frame_*"))[0])
shape_1 = image.shape[0:2][::-1]

Z = 263/1000  # in meters

camera_matrix_l, dist_coefs_l = load_calibration(job_json, "cam_0")
camera_matrix_r, dist_coefs_r = load_calibration(job_json, "cam_1")
R, T, E, F = load_stereo_calibration(job_json)

frame_num = "frame_10628.png"

# cam_0 = cv2.imread(str(data_dir / "processed" / "cam_0" / "frame_500.png"))
cam_1 = cv2.imread(str(data_dir / "processed" / "cam_1" / "frame_10628.png"))

# callback_state = {"mouse_pos_l": (-1, -1), "mouse_pos_r": (-1, -1)}
# win_name_0 = "left"
# cv2.namedWindow(win_name_0)

# win_name_1 = "right"
# cv2.namedWindow(win_name_1)
# cv2.setMouseCallback(win_name_1, on_mouse_left, callback_state)

point_r_1 = (911, 510)
point_r_2 = (720, 696)

cv2.rectangle(cam_1, point_r_1, point_r_2, (0, 0, 255), 2)

img_pt, _ = transfer_bbox_right_left_dist(camera_matrix_l, dist_coefs_l, camera_matrix_r, dist_coefs_r, R, T, Z, point_r_1)
point_l_1 = tuple(img_pt.reshape(2).astype(np.int64))

img_pt, _ = transfer_bbox_right_left_dist(camera_matrix_l, dist_coefs_l, camera_matrix_r, dist_coefs_r, R, T, Z, point_r_2)
point_l_2 = tuple(img_pt.reshape(2).astype(np.int64))

point_l_3 = (point_l_1[0], point_l_1[1] + 720)
point_l_4 = (point_l_2[0], point_l_2[1] + 720)

# cv2.rectangle(cam_0, point_l_1, point_l_2, (0, 0, 255), 1)
# cv2.rectangle(cam_0, point_l_3, point_l_4, (0, 0, 255), 1)

images = glob(str(data_dir / "processed" / "cam_0" / "frame_*.png"))

for image in images:
    cam_0 = cv2.imread(image)
    cv2.rectangle(cam_0, point_l_1, point_l_2, (0, 0, 255), 1)
    cv2.rectangle(cam_0, point_l_3, point_l_4, (0, 0, 255), 1)

    h = max(cam_0.shape[0], cam_1.shape[0])
    w = cam_0.shape[1] + cam_1.shape[1]

    combined = np.zeros((h, w, 3), dtype=np.uint8)

    combined[0:cam_0.shape[0], 0:cam_0.shape[1]] = cam_0
    combined[0:cam_1.shape[0], cam_0.shape[1]:cam_0.shape[1] + cam_1.shape[1]] = cam_1

    cv2.imwrite(str(data_dir / "output" / pathlib.Path(image).name), combined)

# while True:
#     display_0 = cam_0.copy()
#     display_1 = cam_1.copy()
#
#     if 0 <= callback_state['mouse_pos_l'][0] <= shape_0[0] and 0 <= callback_state['mouse_pos_l'][1] <= shape_0[1]:
#         text_0 = f"x={callback_state['mouse_pos_l'][0]}, y={callback_state['mouse_pos_l'][1]}"
#         cv2.putText(
#             display_0,
#             text_0,
#             (10, 25),  # top-left corner
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.7,
#             (255, 255, 255),
#             2,
#             cv2.LINE_AA,
#         )
#         cv2.circle(display_0, callback_state['mouse_pos_l'], 4, (255,0,255), -1)
#
#     if callback_state['mouse_pos_r'][0] >= 0 and callback_state['mouse_pos_r'][1] >= 0:
#         text_1 = f"x={callback_state['mouse_pos_r'][0]}, y={callback_state['mouse_pos_r'][1]}"
#         cv2.putText(
#             display_1,
#             text_1,
#             (10, 25),  # top-left corner
#             cv2.FONT_HERSHEY_SIMPLEX,
#             0.7,
#             (255, 255, 255),
#             2,
#             cv2.LINE_AA,
#         )
#
#     cv2.imshow(win_name_0, display_0)
#     cv2.imshow(win_name_1, display_1)
#
#     if cv2.waitKey(1) & 0xFF == 27:
#         break
#
# cv2.destroyAllWindows()


In [23]:
import os

folder = r"C:\Users\Bas_K\OneDrive\Documenten\School\Maastricht university\wafer_research\datasets\silicon_test_3\output"

files = sorted(
    [f for f in os.listdir(folder) if f.startswith("frame_") and f.endswith(".png")],
    key=lambda x: int(x.split("_")[1].split(".")[0])
)

# First pass: temporary names
temp_names = []
for i, f in enumerate(files):
    old_path = os.path.join(folder, f)
    temp_path = os.path.join(folder, f"tmp_{i:04d}.png")
    os.rename(old_path, temp_path)
    temp_names.append(temp_path)

# Second pass: final names
for i, temp_path in enumerate(temp_names):
    new_path = os.path.join(folder, f"frame_{i:04d}.png")
    os.rename(temp_path, new_path)